# Deploy Metric Views — RC Recon
Meta-driven deployer: discovers all `*.metric_view.yml` fixture files and executes
`CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML` for each.

New metric view = drop a YAML in `fixtures/metric_views/` and re-run this notebook.

In [ ]:
dbutils.widgets.text("catalog_use", "hls_fde", "Catalog")
dbutils.widgets.text("schema_use",  "rc_recon",  "Schema")

In [ ]:
import os
from pathlib import Path

catalog_use = dbutils.widgets.get("catalog_use")
schema_use  = dbutils.widgets.get("schema_use")

# Resolve fixtures directory relative to this notebook
# When deployed via DABs the notebook lands in the workspace alongside the bundle
notebook_dir = Path(os.path.abspath(""))
fixtures_path = notebook_dir.parent / "fixtures" / "metric_views"

print(f"Catalog : {catalog_use}")
print(f"Schema  : {schema_use}")
print(f"Fixtures: {fixtures_path}")

yml_files = sorted(fixtures_path.glob("*.yml"))
print(f"Found {len(yml_files)} metric view YAML file(s)")

In [ ]:
for yml_file in yml_files:
    with open(yml_file, 'r') as f:
        yaml_content = f.read()

    # Substitute {catalog} and {schema} placeholders
    yaml_content = yaml_content.format(catalog=catalog_use, schema=schema_use)

    # Derive view name: mv_claim_payment_variance.metric_view.yml → mv_claim_payment_variance
    view_name      = yml_file.stem.replace(".metric_view", "")
    full_view_name = f"{catalog_use}.{schema_use}.{view_name}"

    create_sql = f"""CREATE OR REPLACE VIEW {full_view_name}
WITH METRICS
LANGUAGE YAML
AS $$
{yaml_content}
$$"""

    print(f"Deploying {full_view_name} ...", end=" ")
    spark.sql(create_sql)
    print("✓")

print("\nAll metric views deployed.")